## **PONTIFICIA UNIVERSIDAD JAVERIANA**
#### **Procesamiento de Alto Volumen de Datos**
**Fecha**: 19 de marzo 2026

**Autor**: Daniel Felipe Castro Moreno

**Tema**: Introducción a cuaderno Python con PySpark:

    - Prueba de servidor Spark en Jupyter
    - Clúster Spark grupal
    - Servidor Jupyter como framework 

**Objetivo**: Introducción al análisis de alto volumen de datos con PySpark de Apache

In [1]:
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable


In [2]:
!pip install matplotlib

Defaulting to user installation because normal site-packages is not writeable


In [3]:
!pip install seaborn

Defaulting to user installation because normal site-packages is not writeable


In [4]:
!pip install findspark

Defaulting to user installation because normal site-packages is not writeable


In [5]:
### Importación de bibliotecas básicas
import os                         #---> Para interacción con el sistema operativo
import sys                        #---> Para recursos del sistema
import pandas as pd               #---> Para graficar objetos y dataframe
import numpy as np                #---> Para algebra matricial
import matplotlib.pyplot as plt   #---> Para formatos de gráficas
import seaborn as sns             #---> Para estadística y graficar

In [6]:
### Importación de bibliotecas especializadas
import findspark                              #---> Para librerias de Apache Spark
findspark.init('/Almacen/Spark')
from pyspark import SparkConf, SparkContext   #---> Contexto y configuración de PySpark
from pyspark.sql import SparkSession          #---> Sesión para entorno de consultas sobre SQL

In [7]:
configuration = SparkConf()
configuration.set("spark.scheduler.mode", "FAIR")
configuration.set("spark.scheduler.allocation", "/mnt/sda1/Cluster/Spark/conf/fairscheduler.xml")
configuration.setMaster("spark://10.43.97.166:7077")
configuration.setAppName("AppStroke_Castro")
sparkCastro = SparkSession.builder.config(conf=configuration).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 08:30:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Actividades a realizar

- Aceso a fichero compartido
- Lectura de CSV: Se empieza con "datos estructurados"
- Cargar en un objeto Dataframe del tipo Spark
- Presentamos los nombres de las columnas
- Se cambian los nombres de las columnas
- Tipo y coherencia de los datos
- Identificación de procedimientos para el tratamiento de datos irregulares
- Estadísticas y gráficas exhaustivas
- Tratamiento de las categorías

#### Acceso a el sistema de trabajo stroke_pyspark.csv

- El archivo se encuentra en el fichero compartido /Almacen
- La idea es que el fichero sea leido por todos los nodos al tiempo (Sistema de Ficheros Compartido)

Notas: dfPy00 es un objeto dataframe que tiene el método show que requiere de argumentos porque de lo contrario tratará de mostrar todos los datos, lo que no es útil en procesamiento de datos a gran escala. ICTUS es la variable dependiente de las demás que se quiere calcular, las otras son independientes. Se debe realizar el casting debido a que no se puede realizar operaciones matemáticas o estadísticas con una cadena de caracteres. Double cuando se requiere mayor cantidad de precisión, Float para decimales, entero cuando no hay decimales.

In [8]:
## Se crea el dataframe para acceder al sistema de fichero csv como un objeto dataframe PySpark
## El acceso se dará desde el sistema de ficheros Hadoop HDFS
dfPy00 = sparkCastro.read.format("csv").option("header", "true").load("stroke_pyspark.csv")
dfPy00.show(5)

+-----+------+---+------------+-------------+------------+-------------+--------------+-----------------+----+---------------+------+
|   id|gender|age|hypertension|heart_disease|ever_married|    work_type|Residence_type|avg_glucose_level| bmi| smoking_status|stroke|
+-----+------+---+------------+-------------+------------+-------------+--------------+-----------------+----+---------------+------+
| 9046|  Male| 67|           0|            1|         Yes|      Private|         Urban|           228.69|36.6|formerly smoked|     1|
|51676|Female| 61|           0|            0|         Yes|Self-employed|         Rural|           202.21| N/A|   never smoked|     1|
|31112|  Male| 80|           0|            1|         Yes|      Private|         Rural|           105.92|32.5|   never smoked|     1|
|60182|Female| 49|           0|            0|         Yes|      Private|         Urban|           171.23|34.4|         smokes|     1|
| 1665|Female| 79|           1|            0|         Yes|Self

**1. Cambio de nombres de columnas**

In [9]:
dfPy00.columns

['id',
 'gender',
 'age',
 'hypertension',
 'heart_disease',
 'ever_married',
 'work_type',
 'Residence_type',
 'avg_glucose_level',
 'bmi',
 'smoking_status',
 'stroke']

In [10]:
### Importar funciones
from pyspark.sql import functions as F

## Copia y cambio
NomOriginal = ['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status', 'stroke']
NomNuevo = ['ID', 'SEX', 'EDAD', 'HIPERT', 'ENF_CARD', 'CASADO', 'TRABAJO', 'RESIDENCIA', 'NIV_GLUCOSA', 'IMC', 'FUMADOR', 'ICTUS']

#### Versión
### original --> nueva "Atención que habrá un cambio"
dfPy01 = dfPy00

### Se hará un comprimido para el cambio en un bucle
for antes, nuevo in zip(dfPy01.columns, NomNuevo):
    dfPy01 = dfPy01.withColumnRenamed(antes, nuevo)
dfPy01.columns

['ID',
 'SEX',
 'EDAD',
 'HIPERT',
 'ENF_CARD',
 'CASADO',
 'TRABAJO',
 'RESIDENCIA',
 'NIV_GLUCOSA',
 'IMC',
 'FUMADOR',
 'ICTUS']

In [11]:
dfPy01.show(5)

[Stage 2:>                                                          (0 + 1) / 1]

+-----+------+----+------+--------+------+-------------+----------+-----------+----+---------------+-----+
|   ID|   SEX|EDAD|HIPERT|ENF_CARD|CASADO|      TRABAJO|RESIDENCIA|NIV_GLUCOSA| IMC|        FUMADOR|ICTUS|
+-----+------+----+------+--------+------+-------------+----------+-----------+----+---------------+-----+
| 9046|  Male|  67|     0|       1|   Yes|      Private|     Urban|     228.69|36.6|formerly smoked|    1|
|51676|Female|  61|     0|       0|   Yes|Self-employed|     Rural|     202.21| N/A|   never smoked|    1|
|31112|  Male|  80|     0|       1|   Yes|      Private|     Rural|     105.92|32.5|   never smoked|    1|
|60182|Female|  49|     0|       0|   Yes|      Private|     Urban|     171.23|34.4|         smokes|    1|
| 1665|Female|  79|     1|       0|   Yes|Self-employed|     Rural|     174.12|  24|   never smoked|    1|
+-----+------+----+------+--------+------+-------------+----------+-----------+----+---------------+-----+
only showing top 5 rows



**2. Tipos y coherencia de datos**

In [12]:
#### -----> Func
dfPy01.printSchema()

root
 |-- ID: string (nullable = true)
 |-- SEX: string (nullable = true)
 |-- EDAD: string (nullable = true)
 |-- HIPERT: string (nullable = true)
 |-- ENF_CARD: string (nullable = true)
 |-- CASADO: string (nullable = true)
 |-- TRABAJO: string (nullable = true)
 |-- RESIDENCIA: string (nullable = true)
 |-- NIV_GLUCOSA: string (nullable = true)
 |-- IMC: string (nullable = true)
 |-- FUMADOR: string (nullable = true)
 |-- ICTUS: string (nullable = true)



In [13]:
### Se cambia el ID, EDAD, HIPERT, ENF_CARD e ICTUS por tipo de dato entero porque son numéricos y no requieren decimales (flotante tiene mayor tamaño)
### Se cambia NIV_GLUCOSA e IMC por dobles dado que son numéricos y contienen decimales
dfPy02 = dfPy01.withColumns({
    "ID": F.col("ID").cast("int"),
    "EDAD": F.col("EDAD").cast("int"),
    "HIPERT": F.col("HIPERT").cast("int"),
    "ENF_CARD": F.col("ENF_CARD").cast("int"),
    "NIV_GLUCOSA": F.col("NIV_GLUCOSA").cast("double"),
    "IMC": F.col("IMC").cast("double"),
    "ICTUS": F.col("ICTUS").cast("int")
})
dfPy02.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- SEX: string (nullable = true)
 |-- EDAD: integer (nullable = true)
 |-- HIPERT: integer (nullable = true)
 |-- ENF_CARD: integer (nullable = true)
 |-- CASADO: string (nullable = true)
 |-- TRABAJO: string (nullable = true)
 |-- RESIDENCIA: string (nullable = true)
 |-- NIV_GLUCOSA: double (nullable = true)
 |-- IMC: double (nullable = true)
 |-- FUMADOR: string (nullable = true)
 |-- ICTUS: integer (nullable = true)



In [14]:
### Se realizan grupos para explícitamente señalar categorías

dfPy02.groupby(["SEX"]).count().show()
dfPy02.groupby(["CASADO"]).count().show()
dfPy02.groupby(["TRABAJO"]).count().show()
dfPy02.groupby(["RESIDENCIA"]).count().show()
dfPy02.groupby(["FUMADOR"]).count().show()

+------+-----+
|   SEX|count|
+------+-----+
|Female| 2994|
| Other|    1|
|  Male| 2115|
+------+-----+

+------+-----+
|CASADO|count|
+------+-----+
|    No| 1757|
|   Yes| 3353|
+------+-----+



+-------------+-----+
|      TRABAJO|count|
+-------------+-----+
| Never_worked|   22|
|Self-employed|  819|
|      Private| 2925|
|     children|  687|
|     Govt_job|  657|
+-------------+-----+



+----------+-----+
|RESIDENCIA|count|
+----------+-----+
|     Urban| 2596|
|     Rural| 2514|
+----------+-----+

+---------------+-----+
|        FUMADOR|count|
+---------------+-----+
|         smokes|  789|
|        Unknown| 1544|
|   never smoked| 1892|
|formerly smoked|  885|
+---------------+-----+



In [15]:
#### Cantidad de Datos Nulos o Irregulares
dfPy02.select([F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c) for c in dfPy02.columns]).show()

+---+---+----+------+--------+------+-------+----------+-----------+---+-------+-----+
| ID|SEX|EDAD|HIPERT|ENF_CARD|CASADO|TRABAJO|RESIDENCIA|NIV_GLUCOSA|IMC|FUMADOR|ICTUS|
+---+---+----+------+--------+------+-------+----------+-----------+---+-------+-----+
|  0|  0|   0|     0|       0|     0|      0|         0|          0|201|      0|    0|
+---+---+----+------+--------+------+-------+----------+-----------+---+-------+-----+



In [16]:
print(f"Cantidad total de registros {dfPy02.count()}")
print(f"Porcentaje registros nulos {201*100/dfPy02.count()}%")

Cantidad total de registros 5110
Porcentaje registros nulos 3.9334637964774952%


### **Análisis preliminar**

- Se observa que la columna IMC tiene aproximadamente el 4% de los datos Nulos
- La categoría género tiene un registro "Other"

Decisiones:

- Se elimina el registro "Other" de género
- Se saca el promedio de IMC para hacer sustitución por los Nulos en las columnas
- El promedio se toma por género, por estratos de edades

**3. Identificación y tratamiento de datos duros**

In [17]:
# Se verifica la columna de género
dfPy02.groupBy(['SEX']).count().show()

+------+-----+
|   SEX|count|
+------+-----+
|Female| 2994|
| Other|    1|
|  Male| 2115|
+------+-----+



In [18]:
# Se elimina el registro de género (SEX) "Other"
dfPy02 = dfPy02.where("SEX <> 'Other'")

# Se verifica el cambio
dfPy02.groupBy(['SEX']).count().show()

+------+-----+
|   SEX|count|
+------+-----+
|Female| 2994|
|  Male| 2115|
+------+-----+



In [25]:
# Promedio de IMC por género "Female" y por edades
# Se toma los rangos de edades: 0-10 || 11-20 || ...
prom10F = dfPy02.where((F.col('SEX') == F.lit('Female')) & (F.col('EDAD') <= 10)).select(F.mean(F.col('IMC'))).collect()
prom20F = dfPy02.where((F.col('SEX') == F.lit('Female')) & (F.col('EDAD') > 10) & (F.col('EDAD') <= 20)).select(F.mean(F.col('IMC'))).collect()
prom30F = dfPy02.where((F.col('SEX') == F.lit('Female')) & (F.col('EDAD') > 20) & (F.col('EDAD') <= 30)).select(F.mean(F.col('IMC'))).collect()
prom40F = dfPy02.where((F.col('SEX') == F.lit('Female')) & (F.col('EDAD') > 30) & (F.col('EDAD') <= 40)).select(F.mean(F.col('IMC'))).collect()
prom50F = dfPy02.where((F.col('SEX') == F.lit('Female')) & (F.col('EDAD') > 40) & (F.col('EDAD') <= 50)).select(F.mean(F.col('IMC'))).collect()
prom60F = dfPy02.where((F.col('SEX') == F.lit('Female')) & (F.col('EDAD') > 50) & (F.col('EDAD') <= 60)).select(F.mean(F.col('IMC'))).collect()
prom70F = dfPy02.where((F.col('SEX') == F.lit('Female')) & (F.col('EDAD') > 60) & (F.col('EDAD') <= 70)).select(F.mean(F.col('IMC'))).collect()
prom80F = dfPy02.where((F.col('SEX') == F.lit('Female')) & (F.col('EDAD') > 70) & (F.col('EDAD') <= 80)).select(F.mean(F.col('IMC'))).collect()
prom90F = dfPy02.where((F.col('SEX') == F.lit('Female')) & (F.col('EDAD') > 80) & (F.col('EDAD') <= 90)).select(F.mean(F.col('IMC'))).collect()

In [26]:
# Prueba
print(f"Prueba de promedio Female < 10 años en IMC = {prom10F}")
print(f"Prueba de promedio Female < 20 años en IMC = {prom20F}")
print(f"Prueba de promedio Female < 30 años en IMC = {prom30F}")
print(f"Prueba de promedio Female < 40 años en IMC = {prom40F}")
print(f"Prueba de promedio Female < 50 años en IMC = {prom50F}")
print(f"Prueba de promedio Female < 60 años en IMC = {prom60F}")
print(f"Prueba de promedio Female < 70 años en IMC = {prom70F}")
print(f"Prueba de promedio Female < 80 años en IMC = {prom80F}")
print(f"Prueba de promedio Female < 90 años en IMC = {prom90F}")

Prueba de promedio Female < 10 años en IMC = [Row(avg(IMC)=18.719565217391306)]
Prueba de promedio Female < 20 años en IMC = [Row(avg(IMC)=25.516000000000012)]
Prueba de promedio Female < 30 años en IMC = [Row(avg(IMC)=28.640163934426226)]
Prueba de promedio Female < 40 años en IMC = [Row(avg(IMC)=31.16508313539193)]
Prueba de promedio Female < 50 años en IMC = [Row(avg(IMC)=30.94269662921346)]
Prueba de promedio Female < 60 años en IMC = [Row(avg(IMC)=31.923094170403576)]
Prueba de promedio Female < 70 años en IMC = [Row(avg(IMC)=30.717080745341597)]
Prueba de promedio Female < 80 años en IMC = [Row(avg(IMC)=29.233229813664597)]
Prueba de promedio Female < 90 años en IMC = [Row(avg(IMC)=28.101428571428578)]


In [27]:
# Promedio de IMC por género "Male" y por edades
# Se toma los rangos de edades: 0-10 || 11-20 || ...
prom10M = dfPy02.where((F.col('SEX') == F.lit('Male')) & (F.col('EDAD') <= 10)).select(F.mean(F.col('IMC'))).collect()
prom20M = dfPy02.where((F.col('SEX') == F.lit('Male')) & (F.col('EDAD') > 10) & (F.col('EDAD') <= 20)).select(F.mean(F.col('IMC'))).collect()
prom30M = dfPy02.where((F.col('SEX') == F.lit('Male')) & (F.col('EDAD') > 20) & (F.col('EDAD') <= 30)).select(F.mean(F.col('IMC'))).collect()
prom40M = dfPy02.where((F.col('SEX') == F.lit('Male')) & (F.col('EDAD') > 30) & (F.col('EDAD') <= 40)).select(F.mean(F.col('IMC'))).collect()
prom50M = dfPy02.where((F.col('SEX') == F.lit('Male')) & (F.col('EDAD') > 40) & (F.col('EDAD') <= 50)).select(F.mean(F.col('IMC'))).collect()
prom60M = dfPy02.where((F.col('SEX') == F.lit('Male')) & (F.col('EDAD') > 50) & (F.col('EDAD') <= 60)).select(F.mean(F.col('IMC'))).collect()
prom70M = dfPy02.where((F.col('SEX') == F.lit('Male')) & (F.col('EDAD') > 60) & (F.col('EDAD') <= 70)).select(F.mean(F.col('IMC'))).collect()
prom80M = dfPy02.where((F.col('SEX') == F.lit('Male')) & (F.col('EDAD') > 70) & (F.col('EDAD') <= 80)).select(F.mean(F.col('IMC'))).collect()
prom90M = dfPy02.where((F.col('SEX') == F.lit('Male')) & (F.col('EDAD') > 80) & (F.col('EDAD') <= 90)).select(F.mean(F.col('IMC'))).collect()

In [29]:
# Prueba
print(f"Prueba de promedio Male < 10 años en IMC = {prom10M}")
print(f"Prueba de promedio Male < 20 años en IMC = {prom20M}")
print(f"Prueba de promedio Male < 30 años en IMC = {prom30M}")
print(f"Prueba de promedio Male < 40 años en IMC = {prom40M}")
print(f"Prueba de promedio Male < 50 años en IMC = {prom50M}")
print(f"Prueba de promedio Male < 60 años en IMC = {prom60M}")
print(f"Prueba de promedio Male < 70 años en IMC = {prom70M}")
print(f"Prueba de promedio Male < 80 años en IMC = {prom80M}")
print(f"Prueba de promedio Male < 90 años en IMC = {prom90M}")

Prueba de promedio Male < 10 años en IMC = [Row(avg(IMC)=19.1022813688213)]
Prueba de promedio Male < 20 años en IMC = [Row(avg(IMC)=25.246982758620685)]
Prueba de promedio Male < 30 años en IMC = [Row(avg(IMC)=28.307100591715987)]
Prueba de promedio Male < 40 años en IMC = [Row(avg(IMC)=31.33171806167401)]
Prueba de promedio Male < 50 años en IMC = [Row(avg(IMC)=32.214760147601474)]
Prueba de promedio Male < 60 años en IMC = [Row(avg(IMC)=31.845664739884388)]
Prueba de promedio Male < 70 años en IMC = [Row(avg(IMC)=31.275619834710742)]
Prueba de promedio Male < 80 años en IMC = [Row(avg(IMC)=29.002314814814806)]
Prueba de promedio Male < 90 años en IMC = [Row(avg(IMC)=27.813333333333333)]


In [30]:
# Recortar los decimales a precisión y ubicar el valor (sale en parcial)
print(f"Precisión con 2 decimales = {round(prom70M[0][0], 2)}")

Precisión con 2 decimales = 31.28


In [36]:
# Se sustituye los valores Nulos en la columan IMC por promedios según género
dfPy02 = dfPy02.withColumn(
    "IMC",
    F.when(
        (dfPy02['SEX'] == 'Female') & 
        (dfPy02['IMC'].isNull() & 
        (dfPy02['EDAD'] <= 10)), 
        round(prom10F[0][0], 2)).otherwise(dfPy02['IMC'])
)
dfPy02 = dfPy02.withColumn(
    "IMC",
    F.when(
        (dfPy02['SEX'] == 'Female') & 
        (dfPy02['IMC'].isNull() & 
        (dfPy02['EDAD'] <= 20)), 
        round(prom20F[0][0], 2)).otherwise(dfPy02['IMC'])
)
dfPy02 = dfPy02.withColumn(
    "IMC",
    F.when(
        (dfPy02['SEX'] == 'Female') & 
        (dfPy02['IMC'].isNull() & 
        (dfPy02['EDAD'] <= 30)), 
        round(prom30F[0][0], 2)).otherwise(dfPy02['IMC'])
)
dfPy02 = dfPy02.withColumn(
    "IMC",
    F.when(
        (dfPy02['SEX'] == 'Female') & 
        (dfPy02['IMC'].isNull() & 
        (dfPy02['EDAD'] <= 40)), 
        round(prom40F[0][0], 2)).otherwise(dfPy02['IMC'])
)
dfPy02 = dfPy02.withColumn(
    "IMC",
    F.when(
        (dfPy02['SEX'] == 'Female') & 
        (dfPy02['IMC'].isNull() & 
        (dfPy02['EDAD'] <= 50)), 
        round(prom50F[0][0], 2)).otherwise(dfPy02['IMC'])
)
dfPy02 = dfPy02.withColumn(
    "IMC",
    F.when(
        (dfPy02['SEX'] == 'Female') & 
        (dfPy02['IMC'].isNull() & 
        (dfPy02['EDAD'] <= 60)), 
        round(prom30F[0][0], 2)).otherwise(dfPy02['IMC'])
)
dfPy02 = dfPy02.withColumn(
    "IMC",
    F.when(
        (dfPy02['SEX'] == 'Female') & 
        (dfPy02['IMC'].isNull() & 
        (dfPy02['EDAD'] <= 70)), 
        round(prom70F[0][0], 2)).otherwise(dfPy02['IMC'])
)
dfPy02 = dfPy02.withColumn(
    "IMC",
    F.when(
        (dfPy02['SEX'] == 'Female') & 
        (dfPy02['IMC'].isNull() & 
        (dfPy02['EDAD'] <= 80)), 
        round(prom80F[0][0], 2)).otherwise(dfPy02['IMC'])
)
dfPy02 = dfPy02.withColumn(
    "IMC",
    F.when(
        (dfPy02['SEX'] == 'Female') & 
        (dfPy02['IMC'].isNull() & 
        (dfPy02['EDAD'] <= 90)), 
        round(prom90F[0][0], 2)).otherwise(dfPy02['IMC'])
)

In [36]:
# Se sustituye los valores Nulos en la columan IMC por promedios según género
dfPy02 = dfPy02.withColumn(
    "IMC",
    F.when(
        (dfPy02['SEX'] == 'Female') & 
        (dfPy02['IMC'].isNull() & 
        (dfPy02['EDAD'] <= 10)), 
        round(prom10F[0][0], 2)).otherwise(dfPy02['IMC'])
)
dfPy02 = dfPy02.withColumn(
    "IMC",
    F.when(
        (dfPy02['SEX'] == 'Female') & 
        (dfPy02['IMC'].isNull() & 
        (dfPy02['EDAD'] <= 20)), 
        round(prom20F[0][0], 2)).otherwise(dfPy02['IMC'])
)
dfPy02 = dfPy02.withColumn(
    "IMC",
    F.when(
        (dfPy02['SEX'] == 'Female') & 
        (dfPy02['IMC'].isNull() & 
        (dfPy02['EDAD'] <= 30)), 
        round(prom30F[0][0], 2)).otherwise(dfPy02['IMC'])
)
dfPy02 = dfPy02.withColumn(
    "IMC",
    F.when(
        (dfPy02['SEX'] == 'Female') & 
        (dfPy02['IMC'].isNull() & 
        (dfPy02['EDAD'] <= 40)), 
        round(prom40F[0][0], 2)).otherwise(dfPy02['IMC'])
)
dfPy02 = dfPy02.withColumn(
    "IMC",
    F.when(
        (dfPy02['SEX'] == 'Female') & 
        (dfPy02['IMC'].isNull() & 
        (dfPy02['EDAD'] <= 50)), 
        round(prom50F[0][0], 2)).otherwise(dfPy02['IMC'])
)
dfPy02 = dfPy02.withColumn(
    "IMC",
    F.when(
        (dfPy02['SEX'] == 'Female') & 
        (dfPy02['IMC'].isNull() & 
        (dfPy02['EDAD'] <= 60)), 
        round(prom30F[0][0], 2)).otherwise(dfPy02['IMC'])
)
dfPy02 = dfPy02.withColumn(
    "IMC",
    F.when(
        (dfPy02['SEX'] == 'Female') & 
        (dfPy02['IMC'].isNull() & 
        (dfPy02['EDAD'] <= 70)), 
        round(prom70F[0][0], 2)).otherwise(dfPy02['IMC'])
)
dfPy02 = dfPy02.withColumn(
    "IMC",
    F.when(
        (dfPy02['SEX'] == 'Female') & 
        (dfPy02['IMC'].isNull() & 
        (dfPy02['EDAD'] <= 80)), 
        round(prom80F[0][0], 2)).otherwise(dfPy02['IMC'])
)
dfPy02 = dfPy02.withColumn(
    "IMC",
    F.when(
        (dfPy02['SEX'] == 'Female') & 
        (dfPy02['IMC'].isNull() & 
        (dfPy02['EDAD'] <= 90)), 
        round(prom90F[0][0], 2)).otherwise(dfPy02['IMC'])
)

**4. Estadísticas generales**

**5. Categorías y cambio sobre el tipo de datos**